# Notebook ١١: BERT والـ Masked LMs

توقع كلمة مخفية، متجهات حسب السياق، تصنيف النوايا بـ BERT، وNER. BERT حقيقي، كل خلية اتشغلت فعلًا.

## ١. الكلمة المخفية (Masked LM)

BERT بيشوف الجملة من الاتجاهين ويتوقع الكلمة المخفية.

In [ ]:
# نخفي التحذيرات
import warnings; warnings.filterwarnings("ignore")
# واجهة جاهزة للمهام
from transformers import pipeline
# موديل BERT لتوقع [MASK]
fill = pipeline("fill-mask", model="bert-base-uncased")
# 3 جمل
for s in ["Mohamed Salah scored a [MASK] in the final.", "The goalkeeper saved the [MASK].", "Arsenal won the [MASK] league title."]:
# أعلى 4 توقعات
    r = fill(s, top_k=4)
# الكلمة واحتمالها
    print(s, "->", [(x["token_str"], round(x["score"], 2)) for x in r])


## ٢. متجهات تتغير حسب السياق

فرق BERT عن Word2Vec: نفس الكلمة بمتجه مختلف حسب الجملة.

In [ ]:
# PyTorch وNumPy
import torch, numpy as np
# tokenizer وموديل BERT
from transformers import AutoTokenizer, AutoModel
# tokenizer
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
# موديل BERT
bert = AutoModel.from_pretrained("bert-base-uncased").eval()
# متجه كلمة جوه جملة
def word_vec(sentence, word):
# نشفّر الجملة
    enc = tok(sentence, return_tensors="pt")
# من غير gradients
    with torch.no_grad():
# متجه لكل توكن
        h = bert(**enc).last_hidden_state[0]
# مكان الكلمة
    pos = tok.convert_ids_to_tokens(enc["input_ids"][0]).index(word)
# متجه الكلمة
    return h[pos].numpy()
# cosine
def cos(a, b):
# تشابه
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
# bank بمعنى ضفة
a = word_vec("he sat on the bank of the river", "bank")
# bank بمعنى بنك
b = word_vec("she deposited cash at the bank", "bank")
# bank بمعنى ضفة
c = word_vec("they walked along the river bank", "bank")
# معنيين مختلفين
print("river vs money bank:", round(cos(a, b), 2))
# نفس المعنى
print("river vs river bank:", round(cos(a, c), 2))


## ٣. BERT كـ feature extractor للنوايا

نفس جمل النوايا بتاعة notebook ٥ (٦٠ جملة كتبناها). الاختبار صغير (١٥ جملة) فالرقم تقريبي.

In [ ]:
# المصفوفات
import numpy as np
# تقسيم
from sklearn.model_selection import train_test_split
# الموديل
from sklearn.linear_model import LogisticRegression
# جمل ونوايا (من كتابتنا)
data = {
# إحصائيات
 "stats": ["how many goals did salah score", "show me haaland stats", "assists for rice this season", "top scorer in the league", "how many minutes did saka play", "give me player points", "who has the most assists", "goals and assists of palmer", "what are the stats for son", "clean sheets for raya", "points per game of watkins", "best player this season"],
# إصابات
 "injury": ["is salah injured", "any injury news for saka", "is haaland fit to play", "who is out with injury", "when will rice return", "ankle injury update", "is he suspended", "knock on the hamstring", "fitness news for the arsenal squad", "is palmer doubtful", "who is ruled out this week", "injured players list"],
# انتقالات
 "transfer": ["did he join a new club", "any transfer news", "who left on loan", "is salah leaving liverpool", "new signing at arsenal", "loan deal for the winger", "who joined chelsea", "player sold this summer", "transfer window rumours", "has he signed a contract", "who moved on loan", "latest signings"],
# مباريات
 "fixtures": ["when is the next match", "who do arsenal play next", "fixtures for this week", "what time is the game", "next gameweek schedule", "when does liverpool play", "show upcoming fixtures", "is there a match tomorrow", "arsenal vs chelsea date", "kick off time today", "next opponent for city", "match calendar"],
# تحيات
 "greeting": ["hello", "hi there", "good morning", "hey bot", "hello how are you", "hi", "good evening", "hey there", "hello again", "morning", "hi friend", "hey"],
# نقفل القاموس
}
# أسماء النوايا
labels = sorted(data)
# الجمل
texts = [t for k in labels for t in data[k]]
# رقم النية
ids = np.array([labels.index(k) for k in labels for _ in data[k]])
# تقسيم تدريب واختبار
Ttr, Tte, ltr, lte = train_test_split(texts, ids, test_size=0.25, random_state=0, stratify=ids)
# متجه لكل جملة: متوسط متجهات التوكنز
def embed(batch):
# نشفّر الدفعة
    enc = tok(batch, padding=True, return_tensors="pt")
# من غير gradients
    with torch.no_grad():
# متجهات التوكنز
        h = bert(**enc).last_hidden_state
# نتجاهل الحشو
    m = enc["attention_mask"].unsqueeze(-1)
# المتوسط
    return ((h * m).sum(1) / m.sum(1)).numpy()
# متجهات التدريب والاختبار
Etr, Ete = embed(Ttr), embed(Tte)
# (جمل، 768)
print(Etr.shape)
# نصنف فوق متجهات BERT
clf = LogisticRegression(max_iter=2000).fit(Etr, ltr)
# الدقة
print("BERT features test accuracy:", round(clf.score(Ete, lte), 2))


In [ ]:
# أسئلة جديدة
for q in ["is rice injured", "when do arsenal play", "hello bot", "salah assists this season", "any news about a new signing"]:
# احتمالات النوايا
    p = clf.predict_proba(embed([q]))[0]
# النية والثقة
    print(q, "->", labels[int(p.argmax())], round(float(p.max()), 2))


## ٤. NER: استخراج الأسماء

BERT متدرب على NER بيحدد الأشخاص والأندية والدول.

In [ ]:
# موديل NER جاهز
ner = pipeline("token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple")
# جملة تجريبية
text = "Mohamed Salah left Liverpool for Real Madrid in Spain."
# لكل كيان مكتشف
for e in ner(text):
# الكلمة ونوعها وثقة الموديل
    print(e["word"], e["entity_group"], round(float(e["score"]), 2))


In [ ]:
# جمل أخرى
for t in ["Haaland scored twice for Manchester City against Arsenal.", "Rice joined Arsenal from West Ham in 2023."]:
# الكيانات المستخرجة
    print([(e["word"], e["entity_group"]) for e in ner(t)])
